# Semana 1: Carga, Transformación y Reporte del Dataset de Restaurantes (WhatsCooking)

¡Hola profesor y compañeros! En este notebook desarrollo el paso a paso de la **Actividad 1**:

1. Importamos las dos clases que creamos en la carpeta `src/clases_po` (`HFDatasetLoader` y `Reportes`).
2. Nos conectamos a Hugging Face para descargar el dataset `MongoDB/whatscooking.restaurants`.
3. Hacemos una revision inicial para ver como vienen los datos y que columnas tenemos.
4. Aplicamos 4 transformaciones con Pandas para limpiar y organizar la informacion:
   - **Transformacion 1:** Elegimos las columnas que nos sirven y les ponemos nombres claros en español.
   - **Transformacion 2:** Filtramos para quedarnos con restaurantes con datos completos, calificacion decente (>= 3.0) y al menos 5 reseñas.
   - **Transformacion 3:** Creamos columnas nuevas (un puntaje de popularidad y una clasificacion de precio en palabras).
   - **Transformacion 4:** Estandarizamos texto en mayusculas/minusculas, completamos vacios y ordenamos la tabla.
5. Guardamos la tabla resultante en un archivo `.csv`.
6. Generamos el reporte resumen en archivo `.txt` con nuestra clase `Reportes` y lo mostramos en pantalla.


## 1. Preparar el entorno e importar nuestras clases

Primero aseguramos la ruta del proyecto para poder importar sin problemas las clases que programamos en `src/clases_po`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Buscamos la raíz del proyecto para importar src.clases_po
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader, Reportes
print("Clases importadas correctamente.")

/home/runner/work/PIntegrado_AE_Grupo-6/PIntegrado_AE_Grupo-6/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Clases importadas correctamente.


## 2. Descargar los datos desde Hugging Face

Usamos nuestra clase `HFDatasetLoader` pasandole el nombre del dataset `MongoDB/whatscooking.restaurants`. Descargamos una muestra de 5.000 filas para poder trabajar de forma rapida y fluida en nuestra maquina.

In [2]:
loader = HFDatasetLoader(
    dataset_name="MongoDB/whatscooking.restaurants",
    split="train",
    max_rows=5000,
)

df_raw = loader.load()
df_raw.head()

Generating train split:   0%|          | 0/25361 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 25361/25361 [00:03<00:00, 8125.75 examples/s]

Generating train split: 100%|██████████| 25361/25361 [00:03<00:00, 8074.62 examples/s]

,restaurant_id,location,_id,review_count,DogsAllowed,embedding,PriceRange,menu,HappyHour,TakeOut,address,sponsored,attributes,borough,OutdoorSeating,name,cuisine,stars
0,40366661,"{'coordinates': [-73.975981, 40.745132], 'type...",{'$oid': '6095a34a7c34416a90d3206b'},10,None,"[-0.14520384, 0.018315623, -0.018330636, -0.10...",1.0,None,None,True,"{'building': '627', 'coord': [-73.975981, 40.7...",NaN,"{'Alcohol': ''none'', 'Ambience': '{'romantic'...",Manhattan,True,Baby Bo'S Burritos,Tex-Mex,2.5
1,40367442,"{'coordinates': [-74.1350211, 40.6369042], 'ty...",{'$oid': '6095a34a7c34416a90d3209e'},62,True,"[-0.11977468, -0.02157107, 0.0038846824, -0.09...",2.0,"[Grilled cheese sandwich, Baked potato, Lasagn...",None,True,"{'building': '17', 'coord': [-74.1350211, 40.6...",NaN,"{'Alcohol': ''beer_and_wine'', 'Ambience': '{'...",Staten Island,True,Buddy'S Wonder Bar,American,3.5
2,40364610,"{'coordinates': [-74.138263, 40.546681], 'type...",{'$oid': '6095a34a7c34416a90d31ff6'},72,None,"[-0.1004329, -0.014882699, -0.033005167, -0.09...",1.0,"[Mozzarella sticks, Mushroom swiss burger, Spi...",None,True,"{'building': '37', 'coord': [-74.138263, 40.54...",NaN,"{'Alcohol': ''none'', 'Ambience': '{'touristy'...",Staten Island,True,Great Kills Yacht Club,American,4.0
3,40365288,"{'coordinates': [-73.970637, 40.751495], 'type...",{'$oid': '6095a34a7c34416a90d32017'},149,None,"[-0.11735515, -0.0397448, -0.0072645755, -0.09...",2.0,"[French fries, Chicken pot pie, Mac & cheese, ...",True,True,"{'building': '842', 'coord': [-73.970637, 40.7...",NaN,"{'Alcohol': None, 'Ambience': '{'touristy': Fa...",Manhattan,True,Keats Restaurant,American,4.0
4,40363151,"{'coordinates': [-73.9998042, 40.7251256], 'ty...",{'$oid': '6095a34a7c34416a90d31fbd'},7,None,"[-0.096541286, -0.009661355, 0.04402167, -0.12...",1.0,"[doughnuts, chocolate chip cookies, chocolate ...",None,True,"{'building': '120', 'coord': [-73.9998042, 40....",NaN,"{'Alcohol': None, 'Ambience': None, 'BYOB': No...",Manhattan,True,Olive'S,Bakery,5.0


## 3. Primer vistazo a los datos recibidos

Revisamos cuantas filas y columnas llegaron, que nombres tienen y que tipo de dato maneja cada una con `.info()`.

In [3]:
print(f"Total de filas cargadas: {len(df_raw)}")
print(f"Total de columnas: {len(df_raw.columns)}")
print("Columnas disponibles:", list(df_raw.columns))
df_raw.info()

Total de filas cargadas: 5000
Total de columnas: 18
Columnas disponibles: ['restaurant_id', 'location', '_id', 'review_count', 'DogsAllowed', 'embedding', 'PriceRange', 'menu', 'HappyHour', 'TakeOut', 'address', 'sponsored', 'attributes', 'borough', 'OutdoorSeating', 'name', 'cuisine', 'stars']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   restaurant_id   5000 non-null   object 
 1   location        5000 non-null   object 
 2   _id             5000 non-null   object 
 3   review_count    5000 non-null   int64  
 4   DogsAllowed     1145 non-null   object 
 5   embedding       5000 non-null   object 
 6   PriceRange      4558 non-null   float64
 7   menu            3232 non-null   object 
 8   HappyHour       1198 non-null   object 
 9   TakeOut         5000 non-null   bool   
 10  address         5000 non-null   object 
 11  sponsor

## 4. Transformaciones con Pandas

Aqui aplicamos las 4 transformaciones que solicita la rubrica de la actividad.

### Transformacion 1: Seleccionar columnas utiles y renombrarlas a español
El dataset original viene con columnas complejas (como vectores de embedding y coordenadas) que no necesitamos para este analisis. Escogemos solo las 9 columnas relevantes y les cambiamos el nombre a español para que sea mucho mas comodo trabajar.

In [4]:
columnas_interes = [
    "restaurant_id",
    "name",
    "borough",
    "cuisine",
    "stars",
    "review_count",
    "PriceRange",
    "OutdoorSeating",
    "TakeOut",
]

df_seleccion = df_raw[columnas_interes].copy()

nombres_columnas = {
    "restaurant_id": "ID_Restaurante",
    "name": "Nombre",
    "borough": "Barrio",
    "cuisine": "Tipo_Cocina",
    "stars": "Calificacion",
    "review_count": "Total_Resenas",
    "PriceRange": "Rango_Precio",
    "OutdoorSeating": "Mesas_Exteriores",
    "TakeOut": "Para_Llevar",
}

df_seleccion.rename(columns=nombres_columnas, inplace=True)
print(f"Dimensiones tras Transformación 1: {df_seleccion.shape}")
df_seleccion.head()

Dimensiones tras Transformación 1: (5000, 9)


,ID_Restaurante,Nombre,Barrio,Tipo_Cocina,Calificacion,Total_Resenas,Rango_Precio,Mesas_Exteriores,Para_Llevar
0,40366661,Baby Bo'S Burritos,Manhattan,Tex-Mex,2.5,10,1.0,True,True
1,40367442,Buddy'S Wonder Bar,Staten Island,American,3.5,62,2.0,True,True
2,40364610,Great Kills Yacht Club,Staten Island,American,4.0,72,1.0,True,True
3,40365288,Keats Restaurant,Manhattan,American,4.0,149,2.0,True,True
4,40363151,Olive'S,Manhattan,Bakery,5.0,7,1.0,True,True


### Transformacion 2: Filtrar datos por calidad y cantidad de reseñas
Para que los resultados sean confiables, filtramos los datos con estas condiciones:
- Que no tengan el Nombre ni el Tipo_Cocina vacio.
- Restaurantes con calificacion igual o mayor a 3.0 estrellas.
- Que tengan minimo 5 reseñas registradas.

In [5]:
condicion = (
    df_seleccion["Nombre"].notna() &
    df_seleccion["Tipo_Cocina"].notna() &
    (df_seleccion["Calificacion"] >= 3.0) &
    (df_seleccion["Total_Resenas"] >= 5)
)

df_filtrado = df_seleccion[condicion].copy()

print(f"Registros antes del filtrado: {len(df_seleccion)}")
print(f"Registros después del filtrado: {len(df_filtrado)}")
df_filtrado.head()

Registros antes del filtrado: 5000
Registros después del filtrado: 4173


,ID_Restaurante,Nombre,Barrio,Tipo_Cocina,Calificacion,Total_Resenas,Rango_Precio,Mesas_Exteriores,Para_Llevar
1,40367442,Buddy'S Wonder Bar,Staten Island,American,3.5,62,2.0,True,True
2,40364610,Great Kills Yacht Club,Staten Island,American,4.0,72,1.0,True,True
3,40365288,Keats Restaurant,Manhattan,American,4.0,149,2.0,True,True
4,40363151,Olive'S,Manhattan,Bakery,5.0,7,1.0,True,True
5,30075445,Morris Park Bake Shop,Bronx,Bakery,4.0,39,2.0,True,True


### Transformacion 3: Crear columnas calculadas
Agregamos dos columnas nuevas para enriquecer los datos:
1. `Puntaje_Popularidad`: Multiplicamos la calificacion por el total de reseñas (`Calificacion * Total_Resenas`). Nos sirve para identificar que restaurantes tienen buena fama y muchas visitas.
2. `Categoria_Precio`: La columna de precio viene con numeros (1, 2, etc.). La convertimos a etiquetas sencillas: Economico, Moderado, Costoso o Gourmet.

In [6]:
# 1. Columna numérica de popularidad ponderada
df_filtrado["Puntaje_Popularidad"] = (df_filtrado["Calificacion"] * df_filtrado["Total_Resenas"]).round(2)

# 2. Columna categórica del rango de precios
mapa_precios = {
    1: "Económico",
    2: "Moderado",
    3: "Costoso",
    4: "Gourmet / Exclusivo",
}
df_filtrado["Categoria_Precio"] = df_filtrado["Rango_Precio"].map(mapa_precios).fillna("No Especificado")

print("Muestra con las nuevas columnas derivadas:")
df_filtrado[["Nombre", "Calificacion", "Total_Resenas", "Puntaje_Popularidad", "Categoria_Precio"]].head()

Muestra con las nuevas columnas derivadas:


,Nombre,Calificacion,Total_Resenas,Puntaje_Popularidad,Categoria_Precio
1,Buddy'S Wonder Bar,3.5,62,217.0,Moderado
2,Great Kills Yacht Club,4.0,72,288.0,Económico
3,Keats Restaurant,4.0,149,596.0,Moderado
4,Olive'S,5.0,7,35.0,Económico
5,Morris Park Bake Shop,4.0,39,156.0,Moderado


### Transformacion 4: Limpieza de texto, manejo de vacios y ordenamiento
Limpiamos los textos de `Tipo_Cocina` y `Barrio` quitando espacios y dejandolos en formato titulo (primera letra mayuscula). Para los campos booleanos (`Mesas_Exteriores`, `Para_Llevar`), rellenamos los vacios con `False`. Por ultimo, ordenamos de mayor a menor popularidad y reiniciamos el indice.

In [7]:
df_transformado = df_filtrado.copy()

# Normalizamos texto
df_transformado["Tipo_Cocina"] = df_transformado["Tipo_Cocina"].astype(str).str.strip().str.title()
df_transformado["Barrio"] = df_transformado["Barrio"].astype(str).str.strip().str.title()

# Tratamiento de valores faltantes en campos lógicos
df_transformado["Mesas_Exteriores"] = df_transformado["Mesas_Exteriores"].fillna(False).astype(bool)
df_transformado["Para_Llevar"] = df_transformado["Para_Llevar"].fillna(False).astype(bool)

# Ordenamiento por popularidad descendente y reseteo de índice
df_transformado.sort_values(by="Puntaje_Popularidad", ascending=False, inplace=True)
df_transformado.reset_index(drop=True, inplace=True)

print(f"Dimensiones finales del DataFrame transformado: {df_transformado.shape}")
df_transformado.head()

Dimensiones finales del DataFrame transformado: (4173, 11)


/tmp/ipykernel_2505/985125713.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_transformado["Mesas_Exteriores"] = df_transformado["Mesas_Exteriores"].fillna(False).astype(bool)


,ID_Restaurante,Nombre,Barrio,Tipo_Cocina,Calificacion,Total_Resenas,Rango_Precio,Mesas_Exteriores,Para_Llevar,Puntaje_Popularidad,Categoria_Precio
0,40386287,Winnie'S Bar,Manhattan,Chinese,4.0,4708,2.0,True,True,18832.0,Moderado
1,40605180,Nocello,Manhattan,Italian,4.0,4011,1.0,True,True,16044.0,Económico
2,40560302,Serafina Fabulous Pizza,Manhattan,Italian,4.5,3555,2.0,True,True,15997.5,Moderado
3,40643680,Kent Theatre,Brooklyn,American,4.0,2522,2.0,True,True,10088.0,Moderado
4,40560108,101 Restaurant And Bar,Brooklyn,Italian,4.0,2302,2.0,True,True,9208.0,Moderado


## 5. Guardar los datos limpios en archivo CSV

Exportamos nuestro DataFrame transformado a un archivo CSV en la carpeta `datos/`.

In [8]:
ruta_csv = Path("datos/whatscooking_restaurants.csv")
ruta_csv.parent.mkdir(parents=True, exist_ok=True)
df_transformado.to_csv(ruta_csv, index=False)
print(f"Archivo guardado exitosamente en: {ruta_csv}")

Archivo guardado exitosamente en: datos/whatscooking_restaurants.csv


## 6. Generar el reporte resumen en archivo .txt

Creamos un objeto de nuestra clase `Reportes` y le enviamos el DataFrame limpio para que genere y guarde el resumen estadistico en la carpeta `reportes/`.

In [9]:
reportes = Reportes()
ruta_reporte = reportes.guardar_reporte(df_transformado, "reportes/whatscooking_restaurants.txt")
print(f"Reporte generado y guardado en: {ruta_reporte}")

Reporte generado y guardado en: reportes/whatscooking_restaurants.txt


## 7. Ver el reporte generado en pantalla

Imprimimos el reporte final para revisar que todo haya quedado completo y verificar las metricas obtenidas.

In [10]:
reportes.imprimir_reporte(df_transformado)

REPORTE GENERAL DEL DATAFRAME
Total de filas: 4173
Total de columnas: 11
Filas duplicadas: 0

Columnas y tipos de datos:
  - ID_Restaurante: object
  - Nombre: object
  - Barrio: object
  - Tipo_Cocina: object
  - Calificacion: float64
  - Total_Resenas: int64
  - Rango_Precio: float64
  - Mesas_Exteriores: bool
  - Para_Llevar: bool
  - Puntaje_Popularidad: float64
  - Categoria_Precio: object

Valores nulos por columna:
  - ID_Restaurante: 0
  - Nombre: 0
  - Barrio: 0
  - Tipo_Cocina: 0
  - Calificacion: 0
  - Total_Resenas: 0
  - Rango_Precio: 400
  - Mesas_Exteriores: 0
  - Para_Llevar: 0
  - Puntaje_Popularidad: 0
  - Categoria_Precio: 0

Estadisticas de columnas numericas:
       Calificacion  Total_Resenas  Rango_Precio  Puntaje_Popularidad
count      4173.000       4173.000      3773.000             4173.000
mean          3.791        109.507         1.630              421.437
std           0.550        209.332         0.596              845.077
min           3.000          5.